# BigBear Orbital Shaker

The BigBear orbital shaker is a plate shaker with a serial (RS-232 / ASCII) interface. A single serial line can drive up to 16 shaker positions -- called **nests** -- wired together in a *daisy chain*, each addressed independently.

| Property | Value |
|---|---|
| Communication | Serial, ASCII line protocol |
| Serial settings | 9600 baud, 8 data bits, no parity, 1 stop bit, no handshake |
| Line terminator | carriage return (`\r`) |
| Speed range | 60--3570 rpm |
| Acceleration | 1--10 (device scale) |
| Nests per chain | 1--16 |

```{warning}
This driver has NOT been tested against hardware in PyLabRobot. The protocol,
line terminator, and reply formats are reconstructed from the vendor software.
If you verify it on a shaker, please open a PR to remove the not-tested warning.
```

## Daisy chaining

Multiple shaker units share one serial connection: the host talks to the first unit, which is wired to the second, and so on down the chain. Each unit is a **nest** with a 1-based address.

Every command is addressed with an `@<id>` prefix, where `<id>` is the nest number written in **hexadecimal** -- so nest 10 is `@A` and nest 16 is `@10`. A command reaches only the nest it names; the others ignore it. This lets one port run up to 16 shakers with independent speed, direction, and start/stop.

At `setup()` the driver:

1. sends `U` to put the chain into daisy-chain (addressing) mode,
2. sends `~` to ask how many units are present, and
3. checks that the reported count matches the `num_shakers` you passed.

If the counts disagree (a unit is off, unplugged, or `num_shakers` is wrong) setup raises a `BigBearError`. Set `num_shakers` to the number of physical units on the chain.

## Physical setup

Connect the shaker's serial port to your computer, typically through a USB-to-serial adapter. Chain any additional units in series and give the whole chain a single port.

Make sure the shaker's serial parameters match the driver defaults (9600 baud, 8 data bits, no parity, 1 stop bit, no handshake).

## Connect

`setup()` opens the serial port, enters daisy-chain mode, and verifies the shaker count. It also logs the not-tested warning. Pass `num_shakers` to match your chain.

In [ ]:
from pylabrobot.big_bear import BigBearOrbitalShaker

shaker = BigBearOrbitalShaker(port="/dev/ttyUSB0", num_shakers=1)  # replace with your port
await shaker.setup()

## Start shaking

`start_shaking()` sets the acceleration, speed, and direction for a nest and starts it. Shaking continues until you stop it. `device_id` selects which nest (default 1).

In [ ]:
await shaker.start_shaking(
    rpm=750,
    acceleration=3,
    sequence="CW",
    device_id=1,
)

## Stop shaking

Stop a single nest with `stop_shaking()`, or every nest on the chain with `stop_all()`.

In [ ]:
await shaker.stop_shaking(device_id=1)
# or, for the whole chain:
# await shaker.stop_all()

## Driving multiple nests

With a multi-unit chain, address each nest by its `device_id`. Each runs independently.

In [ ]:
# Example for a 3-nest chain (create with num_shakers=3)
# await shaker.start_shaking(rpm=500, device_id=1)
# await shaker.start_shaking(rpm=900, sequence="CCW", device_id=2)
# await shaker.start_shaking(rpm=1200, device_id=3)

## Homing

Home a single nest with `find_home()`, or every nest with `home_all()`.

In [ ]:
await shaker.find_home(device_id=1)
# or, for the whole chain:
# await shaker.home_all()

## Device info

Read a nest's serial-number fields (`Y`, `X`, `Z`). The reply format is device-defined; the raw fields are returned joined.

In [ ]:
print("Serial number:", await shaker.get_serial_number(device_id=1))

## Teardown

`stop()` stops every nest and closes the serial connection.

In [ ]:
await shaker.stop()